##  Testing of yt video rendering package

In [5]:
!pip install ensure py-youtube
!mkdir -p NotebookFusion
!touch NotebookFusion/__init__.py

In [6]:
%%writefile NotebookFusion/logger.py
import logging
import sys

logging_str = "[%(asctime)s : %(levelname)s : %(module)s] : %(message)s"

logging.basicConfig(
    level=logging.INFO,
    format=logging_str,
    handlers=[logging.StreamHandler(sys.stdout)]
)

logger = logging.getLogger("NotebookFusion")

Writing NotebookFusion/logger.py


In [7]:
%%writefile NotebookFusion/custom_exception.py
import sys
from types import ModuleType


def error_message_detail(error: Exception, error_detail: ModuleType) -> str:
    _, _, exc_tb = error_detail.exc_info()  # type: ignore
    if exc_tb is not None:
        file_name = exc_tb.tb_frame.f_code.co_filename
        line_number = exc_tb.tb_lineno
        error_message = (
            f"Error occurred in script: [{file_name}] "
            f"at line number: [{line_number}] "
            f"with message: [{str(error)}]"
        )
    else:
        error_message = f"Error: {str(error)}"
    return error_message


class NotebookFusionException(Exception):
    def __init__(self, error_message: Exception, error_detail: ModuleType):
        super().__init__(str(error_message))
        self.error_message = error_message_detail(
            error_message, error_detail=error_detail
        )

    def __str__(self) -> str:
        return self.error_message


class InvalidURLException(Exception):
    """Exception raised for invalid URLs."""
    def __init__(self, message: str = "The provided URL is invalid."):
        self.message = message
        super().__init__(self.message)

    def __str__(self) -> str:
        return self.message

Writing NotebookFusion/custom_exception.py


In [8]:
%%writefile NotebookFusion/youtube.py
from IPython.display import HTML, display
import re
from NotebookFusion.custom_exception import InvalidURLException
from NotebookFusion.logger import logger


def render_youtube_video(url: str, width: int = 780, height: int = 440) -> str:
    """Render a YouTube video in Jupyter notebook."""
    try:
        regex = r"(?:v=|\/)([0-9A-Za-z_-]{11}).*"
        match = re.search(regex, url)

        if not match:
            raise InvalidURLException(f"Invalid YouTube URL: {url}")

        video_id = match.group(1)
        embed_url = f"https://www.youtube-nocookie.com/embed/{video_id}"

        iframe = f"""
        <iframe width="{width}" height="{height}"
        src="{embed_url}"
        title="YouTube video player"
        frameborder="0"
        allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
        referrerpolicy="strict-origin-when-cross-origin"
        allowfullscreen>
        </iframe>
        """

        display(HTML(iframe))
        logger.info(f"Successfully rendered YouTube video for URL: {url}")

        return "success"

    except InvalidURLException:
        raise
    except Exception as e:
        logger.error(f"Error rendering YouTube video: {e}")
        raise InvalidURLException(f"Failed to render YouTube video: {str(e)}")

Writing NotebookFusion/youtube.py


In [9]:
from NotebookFusion.youtube import render_youtube_video

# Test with valid URL
result = render_youtube_video("https://www.youtube.com/watch?v=dQw4w9WgXcQ")
print(f"Result: {result}")

Result: success
